# Branch And Bound 

## Import Library 

In [1]:
import math
import time
import heapq
import numpy as np
from scipy.optimize import linprog
from copy import deepcopy

## Read Data

In [2]:
def read_solomon_file(filepath: str) -> dict:
    with open(filepath, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    # Dòng chứa số xe và sức chứa
    n_vehicles = capacity = None
    data_rows  = []

    i = 0
    while i < len(lines):
        parts = lines[i].split()
        # Tìm dòng "25  200" (vehicle number + capacity)
        if (n_vehicles is None and len(parts) == 2
                and parts[0].isdigit() and parts[1].isdigit()):
            n_vehicles = int(parts[0])
            capacity   = int(parts[1])
        # Dòng dữ liệu khách hàng: 7 số
        elif len(parts) == 7 and parts[0].isdigit():
            data_rows.append([int(v) for v in parts])
        i += 1

    customers = [
        {
            "id":      r[0],
            "x":       r[1],
            "y":       r[2],
            "demand":  r[3], # nhu cau
            "ready":   r[4],
            "due":     r[5],
            "service": r[6],
        }
        for r in data_rows
    ]

    return {
        "n_vehicles": n_vehicles,
        "capacity":   capacity,
        "customers":  customers,   # index 0 = depot
    }

## Build Data

In [ ]:
def build_problem(raw: dict, n_customers: int) -> dict:
    """
    Xây dựng dict bài toán VRPTW từ raw data Solomon.
    
    Trả về dict gồm:
        nodes, depot, customers, demand, ready, due, service,
        dist, capacity, n_vehicles, arcs, n, M_big
    """
    rows  = raw["customers"][: n_customers + 1]   # depot + n_customers
    nodes = [r["id"] for r in rows]

    x_coord = {r["id"]: r["x"]       for r in rows}
    y_coord = {r["id"]: r["y"]       for r in rows}
    demand  = {r["id"]: r["demand"]  for r in rows}
    ready   = {r["id"]: r["ready"]   for r in rows}
    due     = {r["id"]: r["due"]     for r in rows}
    service = {r["id"]: r["service"] for r in rows}

    # c_ij = t_ij = khoảng cách Euclidean (tốc độ = 1)
    dist = {
        (i, j): math.hypot(x_coord[i] - x_coord[j], y_coord[i] - y_coord[j])
        for i in nodes for j in nodes if i != j
    }

    arcs = [(i, j) for i in nodes for j in nodes if i != j]
    n    = len(nodes)   # tổng số nút (depot + khách hàng)

    # Big-M theo công thức: M_ij = b_i + t_ij + s_service_i - a_j
    M_time = {
        (i, j): max(0.0, due[i] + service[i] + dist[(i, j)] - ready[j])
        for (i, j) in arcs
    }

    return {
        "nodes":      nodes,
        "depot":      nodes[0],
        "customers":  nodes[1:],
        "demand":     demand,
        "ready":      ready,
        "due":        due,
        "service":    service,
        "dist":       dist,
        "capacity":   raw["capacity"],
        "n_vehicles": raw["n_vehicles"],
        "arcs":       arcs,
        "n":          n,
        "M_time":     M_time,
    }


## Giải LP Relataxion 

In [4]:
def solve_lp_relaxation(problem: dict,
                         fix_to_zero: set,
                         fix_to_one:  set) -> dict:
    """
    Giải LP Relaxation của mô hình VRPTW tại một nút B&B.

    Mô hình LP:
        Biến: x[a] cho mỗi cung a=(i,j), s[i] thời gian phục vụ, u[i] tải tích lũy
        Hàm mục tiêu: min  sum_a c_a * x_a
        Ràng buộc:
            (1) sum_{j} x_{ij} = 1                    forall i in customers    (mỗi KH phục vụ 1 lần)
            (2) sum_{j} x_{hj} - sum_{i} x_{ih} = 0  forall h in customers    (bảo toàn luồng)
            (3) sum_{j!=0} x_{0j} <= m                                         (số xe tối đa)
            (4) a_i <= s_i <= b_i                     forall i in nodes        (time window)
            (5) s_i + t_ij - s_j + s_service_i <= M_ij*(1-x_ij)    forall (i,j) in arcs    (thứ tự thời gian)
            (6) u_i + q_j - u_j <= Q*(1-x_ij)         forall (i,j) in arcs    (sức chứa)
            (7) q_i <= u_i <= Q                       forall i in customers    (giới hạn tải)
            (8) 0 <= x_ij <= 1                                                 (nới lỏng nguyên)
            (+) x_ij = 0 nếu (i,j) in fix_to_zero  (ràng buộc bổ sung của nút)
            (+) x_ij = 1 nếu (i,j) in fix_to_one   (ràng buộc bổ sung của nút)

    Tham số
    -------
    problem      : dict từ build_problem()
    fix_to_zero  : tập cung (i,j) bị ép x_ij = 0 (nhánh trái)
    fix_to_one   : tập cung (i,j) bị ép x_ij = 1 (nhánh phải)

    Trả về
    ------
    dict: status ('optimal'/'infeasible'), obj, x_val, s_val, u_val
    """
    nodes      = problem["nodes"]
    customers  = problem["customers"]
    depot      = problem["depot"]
    arcs       = problem["arcs"]
    dist       = problem["dist"]
    ready      = problem["ready"]
    due        = problem["due"]
    demand     = problem["demand"]
    service    = problem["service"]
    capacity   = problem["capacity"]
    n_vehicles = problem["n_vehicles"]
    M_time     = problem["M_time"]
    n          = problem["n"]          # số nút
    na         = len(arcs)             # số cung

    # -------------------------------------------------------------------
    # Lập index:
    #   x[a]  : cột 0..na-1          (na biến)
    #   s[i]  : cột na..na+n-1       (n biến)
    #   u[i]  : cột na+n..na+2n-1   (n biến)
    # -------------------------------------------------------------------
    arc_idx  = {a: k    for k, a in enumerate(arcs)}
    node_idx = {v: k    for k, v in enumerate(nodes)}
    n_vars   = na + n + n

    def ix(a):  return arc_idx[a]           # index của x_a
    def is_(v): return na + node_idx[v]     # index của s_v
    def iu(v):  return na + n + node_idx[v] # index của u_v

    # -------------------------------------------------------------------
    # Hàm mục tiêu: min sum c_ij * x_ij   (s, u có hệ số 0)
    # -------------------------------------------------------------------
    c_obj = np.zeros(n_vars)
    for a in arcs:
        c_obj[ix(a)] = dist[a]

    # -------------------------------------------------------------------
    # Bounds
    # -------------------------------------------------------------------
    bounds = [None] * n_vars

    # x_ij in [0, 1]
    for a in arcs:
        lo, hi = 0.0, 1.0
        if a in fix_to_zero: lo = hi = 0.0
        if a in fix_to_one:  lo = hi = 1.0
        bounds[ix(a)] = (lo, hi)

    # s_i in [a_i, b_i]
    for v in nodes:
        bounds[is_(v)] = (float(ready[v]), float(due[v]))

    # u_0 = 0 (depot, không có nhu cầu)
    bounds[iu(depot)] = (0.0, 0.0)
    # u_i in [q_i, Q] cho khách hàng
    for v in customers:
        bounds[iu(v)] = (float(demand[v]), float(capacity))

    # -------------------------------------------------------------------
    # Ràng buộc bằng (equality): A_eq @ z = b_eq
    # (1) sum_{j: j!=i} x_{ij} = 1  forall i in customers
    # (2) sum_{j} x_{hj} - sum_{i} x_{ih} = 0  forall h in customers
    # -------------------------------------------------------------------
    A_eq_rows, b_eq = [], []

    # Ràng buộc (1): mỗi khách hàng được phục vụ đúng 1 lần
    for i in customers:
        row = np.zeros(n_vars)
        for j in nodes:
            if j != i and (i, j) in arc_idx:
                row[ix((i, j))] = 1.0
        A_eq_rows.append(row)
        b_eq.append(1.0)

    # Ràng buộc (2): bảo toàn luồng tại mỗi khách hàng
    for h in customers:
        row = np.zeros(n_vars)
        for j in nodes:
            if j != h and (h, j) in arc_idx:
                row[ix((h, j))] += 1.0   # ra khỏi h
        for i in nodes:
            if i != h and (i, h) in arc_idx:
                row[ix((i, h))] -= 1.0   # vào h
        A_eq_rows.append(row)
        b_eq.append(0.0)

    A_eq = np.array(A_eq_rows) if A_eq_rows else None
    b_eq = np.array(b_eq)

    # -------------------------------------------------------------------
    # Ràng buộc bất đẳng thức (upper bound): A_ub @ z <= b_ub
    # (3) sum_{j!=depot} x_{0j} <= m
    
    # (4) a_i <= s_i <= b_i
    # (5) s_i + t_ij - s_j <= M_ij*(1 - x_ij)
    #     <=> s_i - s_j + M_ij * x_ij <= M_ij - t_ij
    # (6) u_i + q_j - u_j <= Q*(1 - x_ij)
    #     <=> u_i - u_j + Q * x_ij <= Q - q_j
    # -------------------------------------------------------------------
    A_ub_rows, b_ub = [], []

    # Ràng buộc (3): số xe tối đa
    row = np.zeros(n_vars)
    for j in customers:
        if (depot, j) in arc_idx:
            row[ix((depot, j))] = 1.0
    A_ub_rows.append(row)
    b_ub.append(float(n_vehicles))

    # Ràng buộc (5): thứ tự thời gian
    # s_i + t_ij - s_j <= M_ij * (1 - x_ij)
    # => s_i - s_j + M_ij * x_ij <= M_ij - t_ij
    b0 = float(due[depot])
    
    for (i, j) in arcs:
        M_ij  = M_time[(i, j)]
        t_ij  = dist[(i, j)]
        serv_i = float(service[i]) if i != depot else 0.0
        row   = np.zeros(n_vars)

        if j == depot:
            row[is_(i)]     =  1.0
            row[ix((i, j))] =  M_ij
            A_ub_rows.append(row)
            b_ub.append(M_ij + float(due[depot]) - t_ij - serv_i)

        elif i == depot:
            row[is_(i)]     =  1.0
            row[is_(j)]     = -1.0
            row[ix((i, j))] =  M_ij
            A_ub_rows.append(row)
            b_ub.append(M_ij - t_ij)

        else:
            row[is_(i)]     =  1.0
            row[is_(j)]     = -1.0
            row[ix((i, j))] =  M_ij
            A_ub_rows.append(row)
            b_ub.append(M_ij - t_ij - serv_i)
    
    # Ràng buộc (6): sức chứa
    # u_i + q_j - u_j <= Q * (1 - x_ij)
    # => u_i - u_j + Q * x_ij <= Q - q_j
    # NOTE: bỏ qua cung (i, depot) vì u_depot = 0 là giá trị giả,
    #       không đại diện cho tải thực. Khi j=depot: u_i <= Q*(1-x_i0),
    #       nếu x_i0=1 thì u_i<=0, mâu thuẫn với u_i>=demand[i]>0.
    Q = float(capacity)
    for (i, j) in arcs:
        if j == depot:
            continue  # ✅ skip: không áp ràng buộc tải khi quay về depot
        q_j = float(demand[j])
        row = np.zeros(n_vars)
        row[iu(i)]      =  1.0
        row[iu(j)]      = -1.0
        row[ix((i, j))] =  Q
        A_ub_rows.append(row)
        b_ub.append(Q - q_j)

    A_ub = np.array(A_ub_rows) if A_ub_rows else None
    b_ub = np.array(b_ub)

    # -------------------------------------------------------------------
    # Gọi HiGHS solver qua scipy.optimize.linprog
    # -------------------------------------------------------------------
    result = linprog(
        c_obj,
        A_ub=A_ub, b_ub=b_ub,
        A_eq=A_eq, b_eq=b_eq,
        bounds=bounds,
        method="highs",
        options={"disp": False, "time_limit": 1800},
    )

    if result.status != 0:   # không tìm được nghiệm khả thi
        return {"status": "infeasible"}

    # Trích xuất giá trị nghiệm
    z       = result.x
    x_val   = {a: z[ix(a)]   for a in arcs}
    s_val   = {v: z[is_(v)]  for v in nodes}
    u_val   = {v: z[iu(v)]   for v in nodes}

    return {
        "status": "optimal",
        "obj":    result.fun,
        "x_val":  x_val,
        "s_val":  s_val,
        "u_val":  u_val,
    }

## Chọn biến phân nhánh

In [5]:
def select_branching_variable(x_val: dict,
                               fix_to_zero: set,
                               fix_to_one:  set,
                               tol: float = 1e-5):
    """
    Chọn biến x_{ij} phân số để phân nhánh.

    Chiến lược: Most Fractional — chọn biến có giá trị gần 0.5 nhất,
    tức là min |x_ij - 0.5|.

    Tham số
    -------
    x_val       : dict {(i,j): float} nghiệm LP
    fix_to_zero : tập cung đã cố định = 0 (bỏ qua)
    fix_to_one  : tập cung đã cố định = 1 (bỏ qua)
    tol         : ngưỡng phân số

    Trả về
    ------
    (i, j) hoặc None nếu tất cả biến đã nguyên
    """
    best_arc  = None
    best_dist = math.inf

    for arc, val in x_val.items():
        if arc in fix_to_zero or arc in fix_to_one:
            continue
        # Biến phân số: không nguyên trong phạm vi [tol, 1-tol]
        if tol < val < 1.0 - tol:
            d = abs(val - 0.5)
            if d < best_dist:
                best_dist = d
                best_arc  = arc

    return best_arc

## Chích xuất các nghiệm nguyên 

In [6]:
def extract_routes(x_val: dict, problem: dict, tol: float = 1e-5) -> list:
    """
    Từ nghiệm x_val nguyên, trích xuất danh sách tuyến đường.

    Tham số
    -------
    x_val   : dict {(i,j): float}, các x_ij ~1 là cung được chọn
    problem : dict bài toán

    Trả về
    ------
    list of dict {customers, cost}
    """
    depot   = problem["depot"]
    dist    = problem["dist"]

    # Xây dựng đồ thị successor từ x_val
    succ = {}
    for (i, j), v in x_val.items():
        if v > 1.0 - tol:   # x_ij = 1
            succ[i] = j

    routes = []
    visited_depots = set()

    # Mỗi xe xuất phát từ depot
    start_nodes = [j for (i, j), v in x_val.items()
                   if i == depot and v > 1.0 - tol]

    for start in start_nodes:
        if start in visited_depots:
            continue
        route_customers = []
        cur = start
        cost = dist[(depot, start)]
        seen = set()

        while cur != depot:
            if cur in seen:          # vòng lặp bất thường
                break
            seen.add(cur)
            if cur != depot:
                route_customers.append(cur)
            nxt = succ.get(cur)
            if nxt is None:
                break
            cost += dist[(cur, nxt)]
            cur   = nxt

        if route_customers:
            routes.append({"customers": route_customers, "cost": cost})
        visited_depots.add(start)

    return routes

## Branch and Bound 

In [7]:
def branch_and_bound(problem: dict, time_limit: float = 120.0) -> dict:
    """
    Thuật toán Branch and Bound (Best-First Search) cho VRPTW.

    Mỗi nút trên cây B&B lưu:
        - fix_to_zero : set các cung (i,j) bị ép x_ij = 0
        - fix_to_one  : set các cung (i,j) bị ép x_ij = 1
        - lb          : cận dưới (giá trị LP Relaxation của nút này)

    Chiến lược phân nhánh:
        - Chọn biến x_ij phân số gần 0.5 nhất (Most Fractional)
        - Tạo nhánh trái  : x_ij = 0 (cấm cung i→j)
        - Tạo nhánh phải  : x_ij = 1 (bắt buộc cung i→j)

    Tham số
    -------
    problem    : dict từ build_problem()
    time_limit : giới hạn thời gian (giây)

    Trả về
    ------
    dict gồm best_cost, best_routes, nodes_explored, nodes_pruned,
              nodes_pruned_infeasible, nodes_pruned_bound,
              nodes_pruned_optimal, elapsed
    """
    start_time = time.time()

    # -------------------------------------------------------------------
    # Bước 1 — Khởi tạo
    # best_cost = UB = +inf (cận trên, nghiệm nguyên tốt nhất hiện tại)
    # -------------------------------------------------------------------
    best_cost   = math.inf
    best_routes = None

    nodes_explored         = 0
    nodes_pruned_infeasible = 0
    nodes_pruned_bound      = 0
    nodes_pruned_optimal    = 0

    # Heap (priority queue): phần tử = (lb, node_id, fix_zero, fix_one)
    # Best-First: pop nút có lb nhỏ nhất trước
    node_counter = 0
    root_node    = (0.0, node_counter, frozenset(), frozenset())  # lb=0 cho root
    heap         = [root_node]
    heapq.heapify(heap)

    while heap:
        # -------------------------------------------------------------------
        # Bước 2 — Kiểm tra time limit và lấy nút
        # -------------------------------------------------------------------
        if time.time() - start_time > time_limit:
            print(f"    [!] Hết time_limit = {time_limit}s — trả về nghiệm tốt nhất hiện tại.")
            break

        lb, _, fix_zero, fix_one = heapq.heappop(heap)
        nodes_explored += 1

        # -------------------------------------------------------------------
        # Bước 3 — Bounding: giải LP Relaxation tại nút hiện tại
        # -------------------------------------------------------------------
        lp_result = solve_lp_relaxation(problem,
                                         fix_to_zero=set(fix_zero),
                                         fix_to_one =set(fix_one))

        # -------------------------------------------------------------------
        # Bước 4 — Pruning
        # -------------------------------------------------------------------

        # Prune by Infeasibility: LP vô nghiệm
        if lp_result["status"] == "infeasible":
            nodes_pruned_infeasible += 1
            continue

        Z_LP = lp_result["obj"]

        # Prune by Bound: cận dưới >= nghiệm tốt nhất hiện tại
        if Z_LP >= best_cost - 1e-8:
            nodes_pruned_bound += 1
            continue

        # Kiểm tra nghiệm nguyên: tất cả x_ij ∈ {0, 1}
        x_val       = lp_result["x_val"]
        branch_var  = select_branching_variable(x_val, set(fix_zero), set(fix_one))

        # Prune by Optimality: không có biến phân số → nghiệm nguyên
        if branch_var is None:
            if Z_LP < best_cost:
                best_cost   = Z_LP
                best_routes = extract_routes(x_val, problem)
            nodes_pruned_optimal += 1
            continue

        # -------------------------------------------------------------------
        # Bước 5 — Branching: tạo hai nhánh từ biến x_{ij} phân số
        # -------------------------------------------------------------------
        i_b, j_b = branch_var

        # Nhánh trái: x_{ij} = 0 (cấm cung i→j)
        left_zero = fix_zero | frozenset([(i_b, j_b)])
        left_one  = fix_one
        node_counter += 1
        heapq.heappush(heap, (Z_LP, node_counter, left_zero, left_one))

        # Nhánh phải: x_{ij} = 1 (bắt buộc cung i→j)
        right_zero = fix_zero
        right_one  = fix_one | frozenset([(i_b, j_b)])
        node_counter += 1
        heapq.heappush(heap, (Z_LP, node_counter, right_zero, right_one))

    elapsed = time.time() - start_time

    return {
        "best_cost":              best_cost,
        "best_routes":            best_routes,
        "nodes_explored":         nodes_explored,
        "nodes_pruned_infeasible": nodes_pruned_infeasible,
        "nodes_pruned_bound":     nodes_pruned_bound,
        "nodes_pruned_optimal":   nodes_pruned_optimal,
        "elapsed":                elapsed,
        "n_vehicles_used":        len(best_routes) if best_routes else 0,
    }

## Print Result 

In [8]:
def print_result(n: int, result: dict):
    """
    In chi tiết kết quả của một instance.

    Tham số
    -------
    n      : số khách hàng
    result : dict từ branch_and_bound()
    """
    print(f"\n{'='*65}")
    print(f"  Instance C101_n{n:02d}  |  n = {n} khách hàng")
    print(f"{'='*65}")

    if result["best_routes"] is None:
        print("  Không tìm được nghiệm khả thi trong giới hạn thời gian.")
        return

    print(f"  Tổng chi phí (Z*)          : {result['best_cost']:.4f}")
    print(f"  Số xe sử dụng              : {result['n_vehicles_used']}")
    print(f"  Số nút B&B đã duyệt        : {result['nodes_explored']}")
    print(f"  Cắt tỉa (infeasible)       : {result['nodes_pruned_infeasible']}")
    print(f"  Cắt tỉa (by bound)         : {result['nodes_pruned_bound']}")
    print(f"  Cắt tỉa (optimality/int)   : {result['nodes_pruned_optimal']}")
    print(f"  Thời gian chạy             : {result['elapsed']:.4f}s")
    print(f"\n  Chi tiết các tuyến:")
    for idx, route in enumerate(result["best_routes"], 1):
        cust_str = " -> ".join(str(c) for c in route["customers"])
        print(f"    Xe {idx:2d}: 0 -> {cust_str} -> 0"
              f"  (cost = {route['cost']:.4f})")

## Run 

In [12]:
def run_experiments(filepath: str = "C101.txt"):
    """
    Chạy B&B cho C101 với n = 5, 10, 15, 25.

    Với mô hình MILP đầy đủ, số biến và ràng buộc tăng theo O(n^2),
    nên chỉ giải được n nhỏ trong thời gian hợp lý.

    Tham số
    -------
    filepath : đường dẫn tới file C101.txt
    """
    print(f"[*] Đọc dữ liệu từ: {filepath}")
    raw = read_solomon_file(filepath)
    print(f"    Số xe: {raw['n_vehicles']}  |  Sức chứa: {raw['capacity']}")

    configs = [
        #{"n":  5, "time_limit":  1800},
        #{"n": 10, "time_limit":  1800},
        {"n": 15, "time_limit": 1800},
       {"n": 25, "time_limit": 1800},
    ]

    summary = []

    for cfg in configs:
        n  = cfg["n"]
        tl = cfg["time_limit"]

        print(f"\n[*] Giải C101_n{n:02d}  (time_limit={tl}s) ...")
        problem = build_problem(raw, n)
        result  = branch_and_bound(problem, time_limit=tl)
        print_result(n, result)

        summary.append({
            "n":        n,
            "cost":     f"{result['best_cost']:.4f}" if result["best_routes"] else "N/A",
            "vehicles": result["n_vehicles_used"],
            "nodes":    result["nodes_explored"],
            "time_s":   f"{result['elapsed']:.4f}",
        })

    # Bảng tổng hợp
    print(f"\n\n{'='*70}")
    print("  KẾT QUẢ THỰC NGHIỆM — Branch and Bound (MILP) — Solomon C101")
    print(f"{'='*70}")
    header = (f"  {'Instance':<14} {'Z* (tối ưu)':>14} "
              f"{'Số xe':>7} {'Số nút B&B':>12} {'T.gian (s)':>12}")
    print(header)
    print(f"  {'-'*14} {'-'*14} {'-'*7} {'-'*12} {'-'*12}")
    for s in summary:
        inst = f"C101_n{s['n']:02d}"
        print(f"  {inst:<14} {s['cost']:>14} {s['vehicles']:>7}"
              f" {s['nodes']:>12} {s['time_s']:>12}")
    print(f"{'='*70}\n")

In [13]:
filepath = "F:\hoc_ki_2_nam_2025_2026\machine_learning\exercise\daa-project\data\C101.txt"
run_experiments(filepath)

[*] Đọc dữ liệu từ: F:\hoc_ki_2_nam_2025_2026\machine_learning\exercise\daa-project\data\C101.txt
    Số xe: 25  |  Sức chứa: 200

[*] Giải C101_n15  (time_limit=1800s) ...


KeyboardInterrupt: 